# UC09 — Inteligencia sobre Reclamaciones de Clientes

**Objetivo:** Clasificar reclamaciones y analizar sentimiento para detectar problemas sistémicos y riesgo regulatorio.

**Tecnologías:** CORTEX.SENTIMENT · CORTEX.COMPLETE · Streamlit

**Tiempo estimado:** 11 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS RECLAMACIONES_AGUA_DB;

In [ ]:
USE DATABASE RECLAMACIONES_AGUA_DB;

In [ ]:
USE SCHEMA PUBLIC;

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;

In [ ]:
USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Reclamaciones Sintéticas

600 reclamaciones con textos realistas de clientes de agua (calidad, presión, facturación, cortes, obras).

In [ ]:
CREATE OR REPLACE TABLE reclamaciones AS
SELECT
    'REC-' || LPAD(SEQ4()::STRING, 5, '0') AS reclamacion_id,
    DATEADD('day', -UNIFORM(0, 365, RANDOM()), CURRENT_DATE()) AS fecha,
    'AB-' || LPAD(UNIFORM(1, 2000, RANDOM())::STRING, 6, '0') AS abonado_id,
    CASE MOD(SEQ4(), 4) WHEN 0 THEN 'Telefono' WHEN 1 THEN 'Email' WHEN 2 THEN 'Web' ELSE 'Oficina' END AS canal,
    'Zona_' || MOD(SEQ4(), 10) AS zona,
    CASE MOD(SEQ4(), 5)
        WHEN 0 THEN 'Llevo 3 días con el agua turbia y con mal sabor. Esto es inaceptable, exijo una solución inmediata y una compensación en la factura.'
        WHEN 1 THEN 'La presión del agua es muy baja desde hace una semana. No puedo ni ducharme. Ya he llamado 3 veces y nadie soluciona nada.'
        WHEN 2 THEN 'Mi última factura es el doble de lo normal y no he cambiado mis hábitos de consumo. Creo que el contador está averiado. Solicito revisión.'
        WHEN 3 THEN 'Nos han cortado el agua sin previo aviso para hacer obras en la calle. Llevamos 8 horas sin suministro. Voy a denunciar ante el Defensor del Ciudadano.'
        ELSE 'Las obras de renovación de tuberías llevan 3 meses y no avanzan. El ruido es insoportable y hay polvo por todas partes. Exijo información sobre el plazo.'
    END AS texto_reclamacion,
    CASE MOD(SEQ4(), 5) WHEN 0 THEN 'Calidad_Agua' WHEN 1 THEN 'Presion_Baja' WHEN 2 THEN 'Facturacion' WHEN 3 THEN 'Corte_Suministro' ELSE 'Obras' END AS tipo_real,
    CASE MOD(SEQ4(), 3) WHEN 0 THEN 'Abierta' WHEN 1 THEN 'Resuelta' ELSE 'Escalada' END AS estado
FROM TABLE(GENERATOR(ROWCOUNT => 600));

In [ ]:
SELECT tipo_real, COUNT(*) AS reclamaciones, estado FROM reclamaciones GROUP BY tipo_real, estado ORDER BY 2 DESC;

## Paso 3 — Análisis de Sentimiento y Clasificación

Usamos Cortex AI para scoring emocional y categorización automática.

In [ ]:
CREATE OR REPLACE TABLE reclamaciones_analisis AS
SELECT *,
    SNOWFLAKE.CORTEX.SENTIMENT(texto_reclamacion) AS sentimiento_score,
    CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto_reclamacion) < -0.5 THEN 'Muy_Negativo'
         WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto_reclamacion) < 0 THEN 'Negativo'
         WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto_reclamacion) < 0.3 THEN 'Neutro'
         ELSE 'Positivo' END AS sentimiento_label,
    CASE WHEN LOWER(texto_reclamacion) LIKE '%defensor%' OR LOWER(texto_reclamacion) LIKE '%denunci%' OR LOWER(texto_reclamacion) LIKE '%abogad%' OR LOWER(texto_reclamacion) LIKE '%juzgad%'
         THEN TRUE ELSE FALSE END AS riesgo_regulatorio
FROM reclamaciones;

In [ ]:
SELECT sentimiento_label, COUNT(*) AS n, COUNT(CASE WHEN riesgo_regulatorio THEN 1 END) AS con_riesgo_regulatorio FROM reclamaciones_analisis GROUP BY 1;

## Paso 4 — Dashboard de Reclamaciones

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()

st.title("📢 Inteligencia sobre Reclamaciones de Clientes")

col1, col2, col3, col4 = st.columns(4)
stats = session.sql("SELECT COUNT(*) AS total, ROUND(AVG(sentimiento_score),2) AS sent_medio, COUNT(CASE WHEN riesgo_regulatorio THEN 1 END) AS riesgo, COUNT(CASE WHEN estado='Abierta' THEN 1 END) AS abiertas FROM reclamaciones_analisis").collect()[0]
col1.metric("Total Reclamaciones", stats["TOTAL"])
col2.metric("Sentimiento Medio", stats["SENT_MEDIO"])
col3.metric("Riesgo Regulatorio", stats["RIESGO"])
col4.metric("Abiertas", stats["ABIERTAS"])

st.subheader("Distribución por Tipo")
df = session.sql("SELECT tipo_real, COUNT(*) AS n FROM reclamaciones_analisis GROUP BY 1 ORDER BY 2 DESC").to_pandas()
st.bar_chart(df.set_index("TIPO_REAL"))

## Paso 5 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS RECLAMACIONES_AGUA_DB;